In [7]:
# %pip install imblearn
%pip install -U imbalanced-learn

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [8]:
import pandas as pd
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from imblearn.over_sampling import SMOTE

In [9]:
# Generate an imbalanced dataset
X, y = make_classification(n_samples=1000, n_features=20, n_informative=2, n_redundant=10,
                           n_clusters_per_class=1, weights=[0.99], flip_y=0, random_state=42)

# Convert to DataFrame for better visualization
df = pd.DataFrame(X, columns=[f'feature_{i}' for i in range(X.shape[1])])
df['target'] = y

# Display first few rows
print(df.head())

   feature_0  feature_1  feature_2  feature_3  feature_4  feature_5  \
0  -0.429244  -2.211862   0.189756   0.588553   0.820374  -0.180392   
1  -0.045512  -2.084113  -3.189191  -0.424236  -2.472718   0.508269   
2   0.252195   1.617045   1.565132  -1.970309   2.048682   0.509295   
3   1.725694  -0.516117   2.210866   0.121844   2.667175  -1.059212   
4  -0.749416   1.106232  -0.664455   0.337766  -0.054664   0.552905   

   feature_6  feature_7  feature_8  feature_9  ...  feature_11  feature_12  \
0  -1.150654   1.471709   0.701585  -0.833474  ...    0.702818   -0.225999   
1  -2.850971   0.911854   0.472527  -1.210002  ...   -2.126279    1.908717   
2  -0.238676   1.445465   0.673300  -0.529416  ...    1.758406   -1.076195   
3   0.107509   1.527901   0.705332  -0.442889  ...    2.289787   -1.482342   
4  -1.497095   1.233776   0.597581  -0.871460  ...   -0.048795    0.320768   

   feature_13  feature_14  feature_15  feature_16  feature_17  feature_18  \
0    1.533434    1.372848  

In [10]:
# Separate features and target
X = df.drop('target', axis=1)
y = df['target']

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [11]:
# Apply SMOTE
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

In [12]:
# Train a RandomForestClassifier
rf = RandomForestClassifier(random_state=42)
rf.fit(X_train_smote, y_train_smote)

# Make predictions
y_pred = rf.predict(X_test)

# Evaluate the model
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00       299
           1       1.00      1.00      1.00         1

    accuracy                           1.00       300
   macro avg       1.00      1.00      1.00       300
weighted avg       1.00      1.00      1.00       300

[[299   0]
 [  0   1]]


In [13]:
# Define the parameter grid
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_features': ['auto', 'sqrt', 'log2'],
    'max_depth': [4, 6, 8, 10],
    'criterion': ['gini', 'entropy']
}

# Apply GridSearchCV
grid_search = GridSearchCV(estimator=rf, param_grid=param_grid, cv=5, n_jobs=-1, verbose=2)
grid_search.fit(X_train_smote, y_train_smote)

# Get the best parameters
best_params = grid_search.best_params_
print(f"Best parameters: {best_params}")

Fitting 5 folds for each of 72 candidates, totalling 360 fits


C:\Users\comp\AppData\Roaming\Python\Python313\site-packages\sklearn\model_selection\_validation.py:528: FitFailedWarning: 
120 fits failed out of a total of 360.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
28 fits failed with the following error:
Traceback (most recent call last):
  File "C:\Users\comp\AppData\Roaming\Python\Python313\site-packages\sklearn\model_selection\_validation.py", line 866, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\comp\AppData\Roaming\Python\Python313\site-packages\sklearn\base.py", line 1382, in wrapper
    estimator._validate_params()
    ~~~~~~~~~~~~~~~~~~~~~~~~~~^^
  File "C:\Users\comp\AppData\Roaming\Python\Py

Best parameters: {'criterion': 'gini', 'max_depth': 4, 'max_features': 'sqrt', 'n_estimators': 100}


In [14]:
# Train the model with best parameters
best_rf = RandomForestClassifier(**best_params, random_state=42)
best_rf.fit(X_train_smote, y_train_smote)

# Make predictions
y_pred_best = best_rf.predict(X_test)

# Evaluate the model
print(classification_report(y_test, y_pred_best))
print(confusion_matrix(y_test, y_pred_best))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00       299
           1       1.00      1.00      1.00         1

    accuracy                           1.00       300
   macro avg       1.00      1.00      1.00       300
weighted avg       1.00      1.00      1.00       300

[[299   0]
 [  0   1]]
